# RAG Handyman — база знаний по ремонту

**Источники:** iFixit API · WikiHow · Mastergrad · Bob Vila  
**Эмбеддинги:** `intfloat/e5-large`  
**Индекс:** FAISS IndexFlatIP

### Быстрый старт
- **Первый запуск:** все ячейки сверху вниз
- **Повторный запуск:** только ячейку в разделе 7

## 0. Установка зависимостей

In [ ]:
!pip install -q transformers sentence-transformers beautifulsoup4 requests tqdm

# faiss-gpu не публикуется на PyPI — ставим нужную версию
import torch
if torch.cuda.is_available():
    # Kaggle / Colab GPU: сначала conda (нативная GPU-сборка), фолбэк на cpu
    !conda install -y -q -c pytorch -c nvidia faiss-gpu 2>/dev/null || pip install -q faiss-cpu
else:
    !pip install -q faiss-cpu

import faiss
print(f'faiss {faiss.__version__} | GPU: {torch.cuda.is_available()}')

In [ ]:
import requests
from bs4 import BeautifulSoup
import re, time, pickle
from pathlib import Path
from tqdm import tqdm
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Optional

# Папка для хранения индекса
INDEX_DIR     = Path("rag_store");  INDEX_DIR.mkdir(exist_ok=True)
INDEX_PATH    = INDEX_DIR / "index.faiss"
CHUNKS_PATH   = INDEX_DIR / "chunks.pkl"
METADATA_PATH = INDEX_DIR / "metadata.pkl"

## 2. Парсеры

### 2.1 iFixit API

In [ ]:
IFIXIT_BASE = 'https://www.ifixit.com/api/2.0'
IFIXIT_HEADERS = {'User-Agent': 'RAG-Handyman/1.0'}


def ifixit_get(endpoint: str, params: dict = None) -> Optional[dict]:
    """GET к публичному iFixit API."""
    try:
        resp = requests.get(
            f'{IFIXIT_BASE}{endpoint}',
            headers=IFIXIT_HEADERS, params=params, timeout=15
        )
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.HTTPError:
        print(f'[iFixit] HTTP {resp.status_code} — {endpoint}'); return None
    except Exception as e:
        print(f'[iFixit] Ошибка: {e}'); return None


def get_ifixit_guides(category: str) -> List[dict]:
    """
    Список гайдов через /wikis/CATEGORY/<category>.
    Категория в формате 'Washing_Machine' (через подчёркивание).
    Возвращает список объектов с guideid и title.
    """
    data = ifixit_get(f'/wikis/CATEGORY/{category}')
    if not data:
        return []
    guides = data.get('guides', [])
    # Иногда гайды вложены по подкатегориям в category_lists
    if not guides:
        for cl in data.get('category_lists', []):
            guides.extend(cl.get('guides', []))
    return guides


def get_ifixit_guide(guide_id: int) -> Optional[dict]:
    """Полный гайд по ID."""
    return ifixit_get(f'/guides/{guide_id}')


def parse_ifixit_guide(guide: dict) -> str:
    """Заголовок + intro + шаги -> чистый текст."""
    parts = []
    title = guide.get('title', '')
    if title:
        parts.append(f'Guide: {title}')

    intro = guide.get('introduction_rendered') or guide.get('introduction', '')
    if intro:
        parts.append(BeautifulSoup(intro, 'html.parser').get_text(' ').strip())

    for step in guide.get('steps', []):
        step_title = step.get('title', '')
        if step_title:
            parts.append(f"Step {step.get('orderby', '')}: {step_title}")
        for line in step.get('lines', []):
            text = line.get('text', '').strip()
            if text:
                text = re.sub(r'\[([^\]]*?)\|?[^\]]*?\]', r'\1', text)
                text = re.sub(r"[']['][']|['][']", '', text)
                parts.append(text)

    return '\n'.join(filter(None, parts))


def fetch_ifixit_category(category: str, delay: float = 0.7) -> List[Dict]:
    """
    Категория в формате 'Washing_Machine' (через подчёркивание).
    Берёт все гайды из категории — без limit, сколько есть.
    """
    guides_meta = get_ifixit_guides(category)
    if not guides_meta:
        print(f'[iFixit] Категория не найдена или пустая: {category}')
        return []

    results = []
    for meta in tqdm(guides_meta, desc=f'iFixit [{category}]'):
        guide_id = meta.get('guideid')
        if not guide_id:
            continue
        guide = get_ifixit_guide(guide_id)
        if not guide:
            continue
        text = parse_ifixit_guide(guide)
        if text:
            results.append({
                'text': text,
                'source': f'https://www.ifixit.com/Guide/{guide_id}',
                'title': guide.get('title', ''),
                'provider': 'ifixit'
            })
        time.sleep(delay)
    print(f'[iFixit] Загружено: {len(results)} гайдов')
    return results


### 2.2 WikiHow

In [ ]:
WH_HEADERS = {'User-Agent': 'RAG-Handyman/1.0'}


def search_wikihow(query: str, max_results: int = 5) -> List[str]:
    try:
        resp = requests.get(
            'https://www.wikihow.com/wikiHowTo',
            headers=WH_HEADERS, params={'search': query, 'ns': 0}, timeout=15
        )
        resp.raise_for_status()
    except Exception as e:
        print(f'[WikiHow] Поиск "{query}": {e}'); return []

    soup = BeautifulSoup(resp.text, 'html.parser')
    urls = []
    for a in soup.select('a.result_link'):
        href = a.get('href', '')
        if href.startswith('/'): href = 'https://www.wikihow.com' + href
        if href and href not in urls: urls.append(href)
        if len(urls) >= max_results: break
    return urls


def parse_wikihow_article(url: str) -> Optional[Dict]:
    try:
        resp = requests.get(url, headers=WH_HEADERS, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        print(f'[WikiHow] {url}: {e}'); return None

    soup = BeautifulSoup(resp.text, 'html.parser')
    parts = []

    title_tag = soup.find('h1', class_='firstHeading') or soup.find('h1')
    title = title_tag.get_text(' ').strip() if title_tag else ''
    if title: parts.append(f'Article: {title}')

    intro = soup.find('div', id='intro')
    if intro: parts.append(intro.get_text(' ').strip())

    # Два варианта селектора — структура WikiHow менялась
    steps = soup.select('li.steps_list_2 .step') or soup.select('.step')
    for i, step in enumerate(steps, 1):
        for tag in step(['script', 'style', 'figure', 'img', 'video']): tag.decompose()
        text = re.sub(r'\s+', ' ', step.get_text(' ')).strip()
        if text: parts.append(f'Step {i}: {text}')

    for sec_id in ['tips', 'warnings']:
        sec = soup.find('div', id=sec_id)
        if sec:
            text = re.sub(r'\s+', ' ', sec.get_text(' ')).strip()
            if text: parts.append(f'{sec_id.capitalize()}: {text}')

    if len(parts) < 2:
        print(f'[WikiHow] Мало контента: {url}'); return None

    return {'text': '\n'.join(parts), 'source': url, 'title': title, 'provider': 'wikihow'}


def fetch_wikihow_topic(query: str, max_articles: int = 5, delay: float = 1.0) -> List[Dict]:
    results = []
    for url in tqdm(search_wikihow(query, max_articles), desc=f'WikiHow [{query}]'):
        article = parse_wikihow_article(url)
        if article: results.append(article)
        time.sleep(delay)
    print(f'[WikiHow] Загружено: {len(results)} статей')
    return results

### 2.3 Mastergrad

Mastergrad — крупнейший русскоязычный форум по ремонту и технике.
Парсим треды из релевантных разделов: берём заголовок темы + все посты треда.

In [ ]:
MG_BASE = 'https://mastergrad.com'
MG_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36',
    'Accept-Language': 'ru-RU,ru;q=0.9'
}

# Разделы форума релевантные для ремонта
MG_SECTIONS = [
    '/forums/bytovaya-tehnika-i-elektronika/stiralnye-mashiny/',
    '/forums/bytovaya-tehnika-i-elektronika/holodilniki/',
    '/forums/bytovaya-tehnika-i-elektronika/posudomoechnye-mashiny/',
    '/forums/bytovaya-tehnika-i-elektronika/pylesosy/',
    '/forums/otoplenie-vodosnabzhenie-kanalizaciya-i-santehnicheskoe-oborudovanie/santehnika-i-santehnicheskoe-oborudovanie/',
    '/forums/otoplenie-vodosnabzhenie-kanalizaciya-i-santehnicheskoe-oborudovanie/kanalizaciya/',
    '/forums/otoplenie-vodosnabzhenie-kanalizaciya-i-santehnicheskoe-oborudovanie/otoplenie/',
    '/forums/otoplenie-vodosnabzhenie-kanalizaciya-i-santehnicheskoe-oborudovanie/kotly-i-kotelnoe-oborudovanie/',
    '/forums/elektrika-i-slabotochka/elektrika/',
    '/forums/ventilyaciya-i-kondicionirovanie/',
    '/forums/gazosnabzhenie/gazovoe-oborudovanie/',
    '/forums/instrumenty-i-silovoe-oborudovanie/elektroinstrumenty-instrukcii-remont-ekspluataciya/',
]


def get_mg_thread_urls(section_path: str, max_threads: int = 15) -> List[str]:
    """
    Берёт ссылки на треды со страницы раздела.
    Треды выглядят как /forums/tNNNNNN-slug/
    """
    try:
        resp = requests.get(
            MG_BASE + section_path,
            headers=MG_HEADERS, timeout=15
        )
        resp.raise_for_status()
    except Exception as e:
        print(f'[Mastergrad] {section_path}: {e}'); return []

    soup = BeautifulSoup(resp.text, 'html.parser')
    urls = []
    seen = set()

    for a in soup.find_all('a', href=True):
        href = a['href']
        if not href.startswith('http'):
            href = MG_BASE + href
        # Треды: mastergrad.com/forums/tNNNNNN-slug/
        if (re.search(r'/forums/t\d+', href)
                and href not in seen):
            seen.add(href)
            urls.append(href)
        if len(urls) >= max_threads: break

    return urls


def parse_mg_thread(url: str) -> Optional[Dict]:
    """
    Парсит тред форума:
    - заголовок темы
    - все посты (вопрос + ответы)
    Короткие посты (< 30 слов) пропускаем — обычно это 'спасибо' и 'согласен'.
    """
    try:
        resp = requests.get(url, headers=MG_HEADERS, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        print(f'[Mastergrad] {url}: {e}'); return None

    soup = BeautifulSoup(resp.text, 'html.parser')
    parts = []

    # Заголовок темы
    h1 = soup.find('h1')
    title = h1.get_text(' ').strip() if h1 else ''
    if title:
        parts.append(f'Тема: {title}')

    # Посты — ищем контейнеры с текстом
    # Реальные классы Mastergrad: .post-body и .pagetext
    posts_found = soup.select('div.post-body, div.pagetext')

    for post in posts_found:
        for tag in post(['script', 'style', 'blockquote', 'aside']): tag.decompose()
        text = re.sub(r'\s+', ' ', post.get_text(' ')).strip()
        # Фильтруем слишком короткие посты
        if len(text.split()) >= 20:
            parts.append(text)

    if len(parts) < 2: return None

    return {
        'text': '\n'.join(parts),
        'source': url,
        'title': title,
        'provider': 'mastergrad'
    }


def fetch_mastergrad_section(
    section_path: str,
    max_threads: int = 15,
    delay: float = 1.0
) -> List[Dict]:
    thread_urls = get_mg_thread_urls(section_path, max_threads=max_threads)
    if not thread_urls:
        print(f'[Mastergrad] Нет тредов: {section_path}'); return []

    results = []
    label = section_path.rstrip('/').split('/')[-1][:30]
    for url in tqdm(thread_urls, desc=f'Mastergrad [{label}]'):
        doc = parse_mg_thread(url)
        if doc: results.append(doc)
        time.sleep(delay)

    print(f'[Mastergrad] {section_path}: {len(results)} тредов')
    return results


def fetch_mastergrad_all(max_threads_per_section: int = 15, delay: float = 1.0) -> List[Dict]:
    results = []
    for section in MG_SECTIONS:
        results.extend(fetch_mastergrad_section(section, max_threads_per_section, delay))
    print(f'[Mastergrad] Всего загружено: {len(results)} тредов')
    return results


### 2.4 Bob Vila

Bob Vila — профессиональные статьи по ремонту с акцентом на технику и инструкции.

In [ ]:
BV_BASE = 'https://www.bobvila.com'
BV_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36'
}

# Категории Bob Vila по ремонту
BV_CATEGORIES = [
    '/category/plumbing/',
    '/category/electrical/',
    '/category/repair-maintenance/',
    '/category/hvac/',
    '/category/doors/',
    '/category/windows/',
    '/category/flooring/',
    '/category/roofing/',
    '/category/basements/',
    '/category/bath-fixtures-fittings/',
    '/category/kitchen/',
    '/category/lawn-and-garden/',
]


def get_bobvila_article_urls(category_path: str, max_results: int = 10) -> List[str]:
    """Берёт ссылки на статьи со страницы категории."""
    try:
        resp = requests.get(
            BV_BASE + category_path,
            headers=BV_HEADERS, timeout=15
        )
        resp.raise_for_status()
    except Exception as e:
        print(f'[BobVila] {category_path}: {e}'); return []

    soup = BeautifulSoup(resp.text, 'html.parser')
    urls = []
    seen = set()

    for a in soup.find_all('a', href=True):
        href = a['href']
        if not href.startswith('http'):
            href = BV_BASE + href
        if ('bobvila.com/articles/' in href
                and not href.endswith('/articles/')
                and href not in seen):
            seen.add(href)
            urls.append(href)
        if len(urls) >= max_results: break

    return urls


def parse_bobvila_article(url: str) -> Optional[Dict]:
    try:
        resp = requests.get(url, headers=BV_HEADERS, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        print(f'[BobVila] {url}: {e}'); return None

    soup = BeautifulSoup(resp.text, 'html.parser')
    h1 = soup.find('h1')
    if not h1: return None

    parts = [f'Article: {h1.get_text(" ").strip()}']
    content = (
        soup.select_one('div.article-body')
        or soup.select_one('div.comp.article-body')
        or soup.select_one('article')
        or soup.select_one('main')
    )
    if not content: return None
    for tag in content(['script', 'style', 'aside', 'nav', 'figure', 'iframe']): tag.decompose()
    for el in content.find_all(['h2', 'h3', 'p', 'li']):
        text = re.sub(r'\s+', ' ', el.get_text(' ')).strip()
        if len(text) > 30: parts.append(text)

    if len(parts) < 2: return None
    return {'text': '\n'.join(parts), 'source': url,
            'title': parts[0].replace('Article: ', ''), 'provider': 'bobvila'}


def fetch_bobvila_categories(max_per_category: int = 10, delay: float = 1.0) -> List[Dict]:
    """Обходит все категории и парсит статьи."""
    results = []
    for cat in BV_CATEGORIES:
        urls = get_bobvila_article_urls(cat, max_results=max_per_category)
        for url in tqdm(urls, desc=f'BobVila [{cat.split("/")[2][:25]}]'):
            doc = parse_bobvila_article(url)
            if doc: results.append(doc)
            time.sleep(delay)
    print(f'[BobVila] Загружено: {len(results)} статей')
    return results


## 3. Чанкование

In [ ]:
def chunk_text(text: str, chunk_size: int = 400, overlap: int = 80, min_chunk_words: int = 30) -> List[str]:
    """
    Разбивает текст на чанки с перекрытием.
    Короткий хвост (< min_chunk_words) присоединяется к предыдущему чанку.
    """
    words = text.split()
    if not words: return []
    step = chunk_size - overlap
    if step <= 0: raise ValueError('overlap должен быть меньше chunk_size')

    chunks = []
    for i in range(0, len(words), step):
        chunk_words = words[i: i + chunk_size]
        if len(chunk_words) < min_chunk_words:
            if chunks: chunks[-1] += ' ' + ' '.join(chunk_words)
            break
        chunks.append(' '.join(chunk_words))
    return chunks


def documents_to_chunks(documents: List[Dict], chunk_size: int = 400, overlap: int = 80) -> tuple:
    """Документы → (all_chunks, metadata). Префикс 'passage: ' обязателен для e5-large."""
    all_chunks, metadata = [], []
    for doc in documents:
        for chunk in chunk_text(doc['text'], chunk_size=chunk_size, overlap=overlap):
            all_chunks.append('passage: ' + chunk)
            metadata.append({'source': doc['source'], 'title': doc.get('title', ''), 'provider': doc.get('provider', 'unknown')})
    print(f'Итого чанков: {len(all_chunks)}')
    return all_chunks, metadata

## 4. Сбор данных

Добавляй / убирай категории и запросы под свои нужды.

In [ ]:
IFIXIT_CATEGORIES = [
    # ── Крупная бытовая техника ───────────────────────────────
    'Washing_Machine', 'Dryer', 'Dishwasher', 'Refrigerator', 'Freezer',
    'Oven', 'Microwave_Oven', 'Electric_Stove', 'Gas_Stove', 'Range_Hood',
    'Air_Conditioner', 'Water_Heater', 'Heat_Pump', 'Furnace',
    # ── Малая кухонная техника ────────────────────────────────
    'Coffee_Maker', 'Espresso_Machine', 'Blender', 'Food_Processor',
    'Toaster', 'Toaster_Oven', 'Electric_Kettle', 'Rice_Cooker',
    'Slow_Cooker', 'Stand_Mixer', 'Juicer', 'Electric_Grill',
    # ── Уборка и уход ─────────────────────────────────────────
    'Vacuum_Cleaner', 'Robot_Vacuum', 'Pressure_Washer',
    'Steam_Cleaner', 'Air_Purifier', 'Humidifier', 'Dehumidifier',
    # ── Сантехника ────────────────────────────────────────────
    'Toilet', 'Faucet', 'Shower', 'Sink',
    'Garbage_Disposal', 'Sump_Pump', 'Water_Softener', 'Water_Filter',
    # ── Электрика и освещение ─────────────────────────────────
    'Lamp', 'Ceiling_Fan', 'Smoke_Detector',
    'Carbon_Monoxide_Detector', 'Doorbell', 'Thermostat', 'Garage_Door_Opener',
    # ── Инструменты ───────────────────────────────────────────
    'Power_Tool', 'Drill', 'Circular_Saw', 'Jigsaw', 'Sander',
    'Angle_Grinder', 'Lawn_Mower', 'Leaf_Blower', 'Chainsaw', 'Generator',
    # ── Электроника ───────────────────────────────────────────
    'Television', 'Monitor', 'Laptop', 'Desktop_Computer',
    'Printer', 'Router', 'Speaker', 'Headphones', 'Tablet',
    # ── Климат ────────────────────────────────────────────────
    'Electric_Fan', 'Space_Heater',
    'Portable_Air_Conditioner', 'Window_Air_Conditioner',
    # ── Разное ────────────────────────────────────────────────
    'Sewing_Machine', 'Bicycle', 'Electric_Scooter',
    'Door_Lock', 'Window',
]

WIKIHOW_QUERIES = [
    'washing machine not draining water', 'washing machine not spinning clothes',
    'washing machine leaking from bottom', 'washing machine making loud noise',
    'dryer not heating up', 'dryer not tumbling', 'dryer taking too long to dry',
    'refrigerator not cooling but freezer works', 'refrigerator making loud noise',
    'refrigerator leaking water inside', 'refrigerator ice maker not working',
    'dishwasher not draining', 'dishwasher leaving residue on dishes',
    'dishwasher not filling with water', 'dishwasher leaking from door',
    'gas stove burner not igniting', 'electric stove burner not heating',
    'oven not reaching temperature', 'microwave not heating food',
    'microwave sparking inside', 'fix leaking faucet dripping',
    'fix running toilet constantly', 'unclog toilet without plunger',
    'unclog bathroom sink drain', 'unclog shower drain hair',
    'fix low water pressure shower', 'water heater not producing hot water',
    'fix leaking pipe under sink', 'garbage disposal not working humming',
    'garbage disposal jammed', 'toilet tank not filling',
    'circuit breaker keeps tripping', 'electrical outlet not working',
    'light switch not working', 'ceiling light flickering',
    'ceiling fan wobbling', 'doorbell not ringing',
    'smoke detector keeps beeping', 'air conditioner not blowing cold air',
    'air conditioner leaking water inside', 'furnace not heating house',
    'thermostat not working', 'fix crack in drywall',
    'repair hole in wall', 'fix squeaky hardwood floor',
    'fix squeaky door hinge', 'fix door not closing properly',
    'fix window not opening', 'repair roof shingle',
    'fix leaking gutter', 'fix garage door not opening',
    'coffee maker not brewing', 'vacuum cleaner lost suction',
    'robot vacuum not charging', 'drill battery not charging',
    'lawn mower not starting', 'pressure washer no pressure',
    'generator not starting',
]

In [ ]:
all_documents: List[Dict] = []

for category in IFIXIT_CATEGORIES:
    all_documents.extend(fetch_ifixit_category(category))

for query in WIKIHOW_QUERIES:
    all_documents.extend(fetch_wikihow_topic(query, max_articles=5))

all_documents.extend(fetch_mastergrad_all(max_threads_per_section=15))

all_documents.extend(fetch_bobvila_categories(max_per_category=10))

print(f'\nВсего документов: {len(all_documents)}')
for provider in ['ifixit', 'wikihow', 'mastergrad', 'bobvila']:
    n = sum(1 for d in all_documents if d['provider'] == provider)
    print(f'  {provider:15s}: {n}')

In [ ]:
all_chunks, metadata = documents_to_chunks(all_documents, chunk_size=400, overlap=80)

## 5. Эмбеддинги

`intfloat/e5-large` требует префиксы:
- `passage: <текст>` — для документов в индексе
- `query: <текст>` — для поисковых запросов

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Устройство: {device}')

model = SentenceTransformer('intfloat/e5-large', device=device)

embeddings = model.encode(
    all_chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

print(f'Эмбеддинги: {embeddings.shape}')

## 6. FAISS-индекс — сборка и сохранение

In [ ]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f'Проиндексировано векторов: {index.ntotal}')

faiss.write_index(index, str(INDEX_PATH))
with open(CHUNKS_PATH,   'wb') as f: pickle.dump(all_chunks, f)
with open(METADATA_PATH, 'wb') as f: pickle.dump(metadata, f)

print(f'Индекс сохранён -> {INDEX_DIR}/')

## 7. Загрузка готового индекса (повторный запуск)

Если `rag_store/` уже есть — запускай только эту ячейку, разделы 4–6 пропускай.

In [ ]:
def load_index():
    missing = [p for p in [INDEX_PATH, CHUNKS_PATH, METADATA_PATH] if not p.exists()]
    if missing:
        raise FileNotFoundError(f'Не найдены файлы: {missing}. Сначала выполни разделы 4-6.')
    idx = faiss.read_index(str(INDEX_PATH))
    with open(CHUNKS_PATH,   'rb') as f: chunks = pickle.load(f)
    with open(METADATA_PATH, 'rb') as f: meta   = pickle.load(f)
    print(f'Загружено: {idx.ntotal} векторов, {len(chunks)} чанков')
    return idx, chunks, meta


# Раскомментировать при повторном запуске (разделы 4-6 не нужны):
# index, all_chunks, metadata = load_index()
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# model  = SentenceTransformer('intfloat/e5-large', device=device)

## 8. Поиск

In [ ]:
def search(query: str, k: int = 5, min_score: float = 0.3) -> List[Dict]:
    q_emb = model.encode(
        ['query: ' + query],
        convert_to_numpy=True, normalize_embeddings=True
    ).astype(np.float32)

    scores, indices = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1: continue
        if float(score) < min_score: continue
        results.append({
            'score':    round(float(score), 4),
            'text':     all_chunks[idx].removeprefix('passage: '),
            'source':   metadata[idx]['source'],
            'title':    metadata[idx]['title'],
            'provider': metadata[idx]['provider']
        })
    return results


# Тест
query = "washing machine won't drain water"
for r in search(query, k=3):
    print(f"[{r['provider'].upper()}] {r['score']} — {r['title']}")
    print(r['text'][:200])
    print('-' * 50)

## 9. Статистика индекса

In [ ]:
from collections import Counter

providers = Counter(m['provider'] for m in metadata)
print('Чанков по источникам:')
for p, n in providers.items():
    print(f'  {p:12s}: {n}')
print(f'\nВсего векторов : {index.ntotal}')
print(f'Размерность    : {index.d}')